In [ ]:
import pandas as pd
import numpy as np

import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
)

import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("data/cleaned_data.csv")

df.head()

In [ ]:
X = df.drop(columns=["stroke","avg_glucose_level"])

y = df["stroke"]


X = pd.get_dummies(
    X,
    drop_first=True
)


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print(X_train.shape)

In [ ]:
tree = DecisionTreeClassifier()

tree.fit(
    X_train_scaled,
    y_train
)

print(
"Train Accuracy:",
accuracy_score(y_train,tree.predict(X_train_scaled))
)

print(
"Test Accuracy:",
accuracy_score(y_test,tree.predict(X_test_scaled))
)

In [ ]:
tree_control = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20
)


tree_control.fit(
    X_train_scaled,
    y_train
)


print(
accuracy_score(
y_test,
tree_control.predict(X_test_scaled)
)
)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)


rf.fit(
    X_train_scaled,
    y_train
)


rf_prob = rf.predict_proba(
    X_test_scaled
)[:,1]


print(
"RF AUC:",
roc_auc_score(
y_test,
rf_prob
)
)

In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)


gb.fit(
X_train_scaled,
y_train
)


gb_prob = gb.predict_proba(
X_test_scaled
)[:,1]


print(
"GB AUC:",
roc_auc_score(
y_test,
gb_prob
)
)

In [ ]:
cv = StratifiedKFold(
n_splits=5,
shuffle=True,
random_state=42
)


models = {

"Logistic Regression":
LogisticRegression(
max_iter=1000,
class_weight="balanced"
),

"Decision Tree":
tree_control,

"Random Forest":
rf,

"Gradient Boosting":
gb

}


for name,model in models.items():

    score = cross_val_score(
        model,
        X_train_scaled,
        y_train,
        cv=cv,
        scoring="roc_auc"
    )

    print(
        name,
        score.mean(),
        score.std()
    )

In [ ]:
pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    RandomForestClassifier(
        random_state=42
    )
)


param_grid={

'randomforestclassifier__n_estimators':
[50,100],

'randomforestclassifier__max_depth':
[5,10,None],

'randomforestclassifier__min_samples_leaf':
[1,5]

}


grid = GridSearchCV(
pipeline,
param_grid,
cv=cv,
scoring="roc_auc",
n_jobs=-1
)


grid.fit(
X_train,
y_train
)


print(grid.best_params_)

print(grid.best_score_)

In [ ]:
best_model = grid.best_estimator_


joblib.dump(
best_model,
"best_model.pkl"
)


print("Model saved")

In [ ]:
loaded_model = joblib.load(
"best_model.pkl"
)


sample = X_test.iloc[:2]


prediction = loaded_model.predict(
sample
)


print(prediction)